# Diff-in-Diff Estimation
Ahora sí puta madre, ya casi vamos a terminar. 

In [9]:
from nbsetup import *
from pipeline import Datos, muestra_comun, COVARIABLES

datos = Datos()
emparejamientos, comunes = muestra_comun(datos)

emp = emparejamientos[300]
emp.pares                          # tratada, control
emp.features.incidentes_asignados.head()  # uid, timestamp, incident_level, peso

,uid,timestamp,incident_level,peso
0,T63,2016-04-22 17:31:48,PIC,1.0
1,T26,2016-04-22 20:00:05,MIN,1.0
2,T98,2016-04-22 14:57:34,PIC,1.0
3,T33,2016-04-22 14:05:15,MIN,1.0
4,T33,2016-04-22 16:31:59,MIN,1.0


## El panel

Antes de estimar nada hay que armar la tabla larga: una fila por unidad y
periodo. Cuatro decisiones de construcción, todas discutibles — si alguna no te
convence, aquí es donde se cambia.

**El día 22 hace las veces de primero de mes.** Las Fotocívicas entraron en vigor
exactamente el 22 de abril de 2019, así que los periodos van del 22 al 22 en vez
de seguir el calendario. Con meses de calendario, abril de 2019 queda mitad pre
y mitad post y hay que decidir qué hacer con ese mes partido; así el corte cae
exacto en la frontera entre `t = -1` y `t = 0`, la ventana queda simétrica
(36 periodos de cada lado) y `t` es directamente tiempo de evento, que es el eje
del event study. Es la misma construcción que usan las ventanas de `Features`.

**Los ceros se ponen.** Un mes sin incidentes es un cero, no una fila ausente.
El panel se arma sobre el producto completo unidad × periodo.

**El conteo es flotante.** Un incidente en área compartida entre dos círculos
tratados entra con 1/2 en cada uno. Sirve para MCO sobre conteos o tasas; para
Poisson habría que replantear el reparto.

**El volumen vehicular es una serie única de toda la ciudad.** No varía entre
unidades, así que los efectos fijos de tiempo lo absorben. Va en el panel solo
por si la variable de resultado se quiere como tasa.


In [2]:
# los periodos: bloques de un mes contados desde el tratamiento
corte = datos.fecha_tratamiento
inicio, fin = datos.ventana

k_min = 0
while corte + pd.DateOffset(months=k_min - 1) >= inicio:
    k_min -= 1

k_max = 0
while corte + pd.DateOffset(months=k_max + 1) <= fin + pd.Timedelta(days=1):
    k_max += 1
k_max -= 1

cortes = pd.DatetimeIndex([corte + pd.DateOffset(months=k) for k in range(k_min, k_max + 2)])

periodos = pd.DataFrame({
    "t": np.arange(k_min, k_max + 1),
    "inicio": cortes[:-1],
    "fin": cortes[1:],
    "post": (np.arange(k_min, k_max + 1) >= 0).astype(int),
})

print(f"{len(periodos)} periodos, t de {periodos.t.min()} a {periodos.t.max()}")
print(f"del {periodos.inicio.iloc[0]:%Y-%m-%d} al {periodos.fin.iloc[-1] - pd.Timedelta(days=1):%Y-%m-%d}")
periodos.query("t in (-36, -1, 0, 35)")

72 periodos, t de -36 a 35
del 2016-04-22 al 2022-04-21


,t,inicio,fin,post
0,-36,2016-04-22,2016-05-22,0
35,-1,2019-03-22,2019-04-22,0
36,0,2019-04-22,2019-05-22,1
71,35,2022-03-22,2022-04-22,1


In [3]:
# las unidades: una fila por unidad emparejada, con su par y su condición
# `par` sirve para efectos fijos de par; el clustering que pidió Alberto es por
# unidad, que es `uid`
pares = emp.pares

unidades = pd.concat([
    pd.DataFrame({"uid": pares.tratada, "par": pares.index, "tratado": 1}),
    pd.DataFrame({"uid": pares.control, "par": pares.index, "tratado": 0}),
], ignore_index=True)

print(f"{len(unidades)} unidades: {unidades.tratado.sum()} tratadas y "
      f"{(1 - unidades.tratado).sum()} controles, en {unidades.par.nunique()} pares")
unidades.head()

186 unidades: 93 tratadas y 93 controles, en 93 pares


,uid,par,tratado
0,T36,0,1
1,T45,1,1
2,T18,2,1
3,T11,3,1
4,T68,4,1


In [11]:
# el volumen vehicular por periodo: cada periodo cae sobre dos meses de
# calendario, así que se promedia ponderando por los días que le tocan a cada uno
volumen_mensual = datos.volumen.assign(mes=lambda x: x.timestamp.dt.to_period("M")).set_index("mes").volumen_mensual

volumen = {}
for _, p in periodos.iterrows():
    dias = pd.date_range(p.inicio, p.fin - pd.Timedelta(days=1), freq="D")
    pesos = pd.Series(dias.to_period("M")).value_counts(normalize=True)
    comunes = pesos.index.intersection(volumen_mensual.index)
    volumen[p.t] = (volumen_mensual.loc[comunes] * pesos.loc[comunes]).sum() / pesos.loc[comunes].sum()

volumen = pd.Series(volumen, name="volumen").rename_axis("t")
volumen.describe().round(2)

count       72.00
mean     15137.88
std       1645.81
min       8805.25
25%      14593.70
50%      15645.96
75%      16129.76
max      17504.04
Name: volumen, dtype: float64

In [6]:
# el panel
NIVELES = None   # None cuenta todos; ("PIC", "FCS") deja solo los que tuvieron
                 # consecuencias personales

asignados = emp.features.incidentes_asignados
dentro = asignados[asignados.uid.isin(unidades.uid)]
if NIVELES is not None:
    dentro = dentro[dentro.incident_level.isin(NIVELES)]

# a qué periodo pertenece cada incidente
indice = np.searchsorted(cortes, dentro.timestamp, side="right") - 1
en_ventana = (indice >= 0) & (indice < len(periodos))
dentro = dentro[en_ventana].assign(t=periodos.t.values[indice[en_ventana]])

malla = pd.MultiIndex.from_product([unidades.uid, periodos.t], names=["uid", "t"])

panel = (
    dentro.groupby(["uid", "t"]).peso.sum()
    .reindex(malla, fill_value=0.0).rename("incidentes").reset_index()
    .merge(unidades, on="uid")
    .merge(periodos[["t", "inicio", "post"]], on="t")
    .merge(volumen, on="t")
    .sort_values(["uid", "t"]).reset_index(drop=True)
)

# verificación: el panel tiene que sumar exactamente lo que entró
descuadre = panel.incidentes.sum() - dentro.peso.sum()

print(f"filas          : {len(panel):,}  ({panel.uid.nunique()} unidades x {len(periodos)} periodos)")
print(f"incidentes     : {panel.incidentes.sum():,.1f}")
print(f"descuadre      : {descuadre:.6f}   <- tiene que ser 0")
print(f"filas en cero  : {(panel.incidentes == 0).mean():.1%}")
panel.head()

filas          : 13,392  (186 unidades x 72 periodos)
incidentes     : 77,102.0
descuadre      : 0.000000   <- tiene que ser 0
filas en cero  : 3.5%


,uid,t,incidentes,par,tratado,inicio,post,volumen
0,C1001,-36,6.0,53,0,2016-04-22,0,15517.130522
1,C1001,-35,8.0,53,0,2016-05-22,0,15177.804094
2,C1001,-34,11.0,53,0,2016-06-22,0,16225.453857
3,C1001,-33,6.0,53,0,2016-07-22,0,16593.199991
4,C1001,-32,6.0,53,0,2016-08-22,0,15447.666857


In [12]:
# las covariables del propensity score, una fila por unidad emparejada.
# Sin interactuar con nada: cuáles entran y cómo se interactúan con el tiempo es
# decisión de la especificación (el punto 3a de Alberto)
covariables = emp.features.matriz.loc[unidades.uid, COVARIABLES]
covariables.head()

,road_length_m,n_vialidades,carriles_ponderados,pct_acceso_controlado,pct_doble_sentido,pct_desnivel,n_niveles,nivel_incidentes,pct_lesionados,tendencia,hubo_fcs,afluencia_nivel
uid,,,,,,,,,,,,
T36,1199.010148,2,5.002458,0.0,1.000000,0.000000,1,6.541667,0.543524,0.166667,1,3.677836e+05
T45,599.290235,1,5.000000,0.0,1.000000,0.000000,1,5.750000,0.541063,-0.291667,1,5.049007e+05
T18,3870.815443,2,3.018694,0.0,0.103133,0.020807,2,9.277778,0.541916,2.833333,1,2.580538e+05
T11,2640.142406,3,3.269653,0.0,0.227616,0.000000,1,13.111111,0.540254,1.833333,1,1.148552e+06
T68,599.349799,1,5.780825,0.0,1.000000,0.000000,1,5.722222,0.444175,-1.458333,0,0.000000e+00
